# 🎵 Music Popularity Predictor — Advanced ML Pipeline

**Upgrades over v1:**
- Feature engineering (artist-level, temporal, derived audio features)
- Model comparison: RandomForest vs XGBoost vs LightGBM vs a Dummy baseline
- Proper 5-fold cross-validation for every model
- Optuna hyperparameter tuning (replaces slow GridSearchCV)
- SHAP explainability — global + per-song
- MLflow experiment tracking
- Best model saved to `models/` for the Streamlit app

## 0. Install & imports

In [ ]:
# Run once to install extras
# !pip install xgboost lightgbm optuna shap mlflow

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import mlflow
import mlflow.sklearn
import optuna
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

os.makedirs('models', exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
sns.set_theme(style='darkgrid', palette='muted')
print('All imports OK ✓')

## 1. Load & inspect data

In [ ]:
df = pd.read_csv('Spotify_data.csv')

# Drop unnamed index column if present
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

print(f'Shape: {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

## 2. Feature engineering

In [ ]:
# ── 2a. Derived audio features ──────────────────────────────────────────────
df['energy_valence']      = df['Energy'] * df['Valence']          # upbeat energy
df['dance_energy']        = df['Danceability'] * df['Energy']     # club factor
df['acoustic_energy_gap'] = df['Energy'] - df['Acousticness']     # electric vs organic
df['loudness_norm']       = (df['Loudness'] + 60) / 60            # scale to ~[0,1]
df['speech_ratio']        = df['Speechiness'] / (df['Energy'] + 1e-6)  # rap indicator

# ── 2b. Artist-level features ────────────────────────────────────────────────
# Average popularity of each artist across the dataset
# (proxy for artist 'fame' — powerful predictor)
if 'Artist' in df.columns:
    artist_mean = df.groupby('Artist')['Popularity'].transform('mean')
    artist_count = df.groupby('Artist')['Popularity'].transform('count')
    df['artist_mean_pop']   = artist_mean
    df['artist_song_count'] = artist_count
    print('Artist features added ✓')
else:
    print('No Artist column found — skipping artist features')

# ── 2c. Temporal features ────────────────────────────────────────────────────
if 'Year' in df.columns:
    df['decade'] = (df['Year'] // 10) * 10
    df['is_recent'] = (df['Year'] >= 2015).astype(int)
    print('Temporal features added ✓')

print(f'\nTotal features: {df.shape[1]}')
df.head(3)

## 3. EDA — key relationships

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

top_features = ['Energy', 'Danceability', 'Loudness', 'Acousticness',
                'dance_energy', 'energy_valence']

for i, feat in enumerate(top_features):
    if feat in df.columns:
        axes[i].scatter(df[feat], df['Popularity'], alpha=0.3, s=8, color='steelblue')
        # add trend line
        z = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), 'Popularity'], 1)
        p = np.poly1d(z)
        xs = np.linspace(df[feat].min(), df[feat].max(), 100)
        axes[i].plot(xs, p(xs), color='tomato', linewidth=2)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('Popularity')
        axes[i].set_title(f'Popularity vs {feat}')

plt.suptitle('Feature vs Popularity (with trend lines)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('models/eda_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Train / test split

In [ ]:
BASE_FEATURES = [
    'Energy', 'Valence', 'Danceability', 'Loudness', 'Acousticness',
    'Tempo', 'Speechiness', 'Liveness',
    # engineered
    'energy_valence', 'dance_energy', 'acoustic_energy_gap',
    'loudness_norm', 'speech_ratio'
]

OPTIONAL = ['artist_mean_pop', 'artist_song_count', 'is_recent', 'decade']
FEATURES = BASE_FEATURES + [f for f in OPTIONAL if f in df.columns]

X = df[FEATURES].fillna(0)
y = df['Popularity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features used: {FEATURES}')

## 5. Model comparison with cross-validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

MODELS = {
    'Dummy (mean baseline)': DummyRegressor(strategy='mean'),
    'Random Forest':         RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    'XGBoost':               XGBRegressor(n_estimators=200, random_state=RANDOM_STATE, verbosity=0),
    'LightGBM':              LGBMRegressor(n_estimators=200, random_state=RANDOM_STATE, verbose=-1),
}

results = []
for name, model in MODELS.items():
    cv_r2  = cross_val_score(model, X_train_s, y_train, cv=kf, scoring='r2')
    cv_mae = -cross_val_score(model, X_train_s, y_train, cv=kf, scoring='neg_mean_absolute_error')
    results.append({
        'Model':    name,
        'CV R²':    cv_r2.mean(),
        'CV R² std': cv_r2.std(),
        'CV MAE':   cv_mae.mean(),
    })
    print(f'{name:30s}  R²={cv_r2.mean():.3f} ± {cv_r2.std():.3f}  MAE={cv_mae.mean():.2f}')

results_df = pd.DataFrame(results).sort_values('CV R²', ascending=False)

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
bars = ax.barh(results_df['Model'], results_df['CV R²'], xerr=results_df['CV R² std'],
               color=colors, capsize=4, height=0.5)
ax.set_xlabel('Cross-validated R²')
ax.set_title('Model Comparison (5-fold CV)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nBest model so far:', results_df.iloc[0]['Model'])

## 6. Optuna hyperparameter tuning (best model)

In [ ]:
# Tune XGBoost — usually wins on tabular data
# Swap for LGBMRegressor if LightGBM won step 5

def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'random_state':      RANDOM_STATE,
        'verbosity':         0,
    }
    model = XGBRegressor(**params)
    score = cross_val_score(model, X_train_s, y_train, cv=3, scoring='r2').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest R² from Optuna: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
# Train final model on full training set with best params
best_model = XGBRegressor(**study.best_params, random_state=RANDOM_STATE, verbosity=0)
best_model.fit(X_train_s, y_train)

y_pred = best_model.predict(X_test_s)

test_r2  = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Test R²:   {test_r2:.4f}')
print(f'Test MAE:  {test_mae:.2f}')
print(f'Test RMSE: {test_rmse:.2f}')

# Actual vs Predicted plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.4, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Popularity')
ax.set_ylabel('Predicted Popularity')
ax.set_title(f'Actual vs Predicted  |  R²={test_r2:.3f}  MAE={test_mae:.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('models/actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. SHAP explainability

In [ ]:
# Global SHAP — which features matter most overall?
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_s)

# Convert back to DataFrame for readable feature names
X_test_df = pd.DataFrame(X_test_s, columns=FEATURES)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('Global Feature Importance (SHAP)')
plt.tight_layout()
plt.savefig('models/shap_global.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm — impact direction (positive / negative effect on popularity)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction')
plt.tight_layout()
plt.savefig('models/shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Per-song waterfall — explain one prediction
idx = 0  # change this to any test sample index
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_test_df.iloc[idx],
    feature_names=FEATURES
))
plt.title(f'Why did the model predict {y_pred[idx]:.0f} popularity? (actual: {y_test.iloc[idx]})')
plt.tight_layout()
plt.savefig('models/shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. MLflow experiment logging

In [ ]:
mlflow.set_experiment('music-popularity')

with mlflow.start_run(run_name='xgboost-optuna'):
    # Log params
    mlflow.log_params(study.best_params)
    mlflow.log_param('features', FEATURES)
    mlflow.log_param('n_features', len(FEATURES))

    # Log metrics
    mlflow.log_metric('test_r2',   test_r2)
    mlflow.log_metric('test_mae',  test_mae)
    mlflow.log_metric('test_rmse', test_rmse)

    # Log artifacts
    mlflow.log_artifact('models/actual_vs_predicted.png')
    mlflow.log_artifact('models/shap_global.png')
    mlflow.log_artifact('models/shap_beeswarm.png')

    # Log model
    mlflow.sklearn.log_model(best_model, 'model')

print('Run logged to MLflow ✓')
print('Run: mlflow ui  to open the dashboard')

## 9. Save artifacts for Streamlit app

In [ ]:
joblib.dump(best_model, 'models/best_model.pkl')
joblib.dump(scaler,     'models/scaler.pkl')

# Save feature list so the app knows what to expect
import json
with open('models/features.json', 'w') as f:
    json.dump(FEATURES, f)

# Save results summary
summary = {
    'model':       'XGBoost (Optuna-tuned)',
    'test_r2':     round(test_r2, 4),
    'test_mae':    round(test_mae, 2),
    'test_rmse':   round(test_rmse, 2),
    'n_features':  len(FEATURES),
    'train_size':  len(X_train),
    'test_size':   len(X_test),
}
with open('models/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
for f in os.listdir('models'):
    print(f'  models/{f}')